# 02D — 30-circle spatial CV, excluding MOS 3–4 (annotation revision)

**Focused sensitivity experiment.** Excludes MOS 3–4 from BOTH model training and circle validation, retaining all 30 other Nadine circles. Reads latest source annotation shapefiles. Results go to a dedicated output folder, preserving original 34-circle NB02D. Default: **native RGB only**, M1 and M2, 3 independent spatial folds = **6 fits**. Optional full illumination treatments can be switched back on in Block 01.

**Scientific caveat:** Excluding MOS 3–4 is a diagnostic scope restriction, not a model-independent reason to discard those annotations. Old 34-circle results must remain available.


## 01 — Environment and source data

In [ ]:
%pip -q install rasterio geopandas pyogrio shapely pandas matplotlib scikit-learn
from pathlib import Path
import json, warnings, math, random, time
import numpy as np, pandas as pd, geopandas as gpd, rasterio, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from rasterio.windows import Window
from rasterio.features import rasterize
from shapely.geometry import box
from shapely.ops import unary_union
from shapely import make_valid
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import average_precision_score, roc_auc_score
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
ROOT=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
                         Path('/content/drive/MyDrive/Musanga/Final')] if p.is_dir()),None)
assert ROOT is not None
BASE_OUT=ROOT/'Outputs/V3/02D_spatial_cv_30_no_MOS34_reannotated';BASE_OUT.mkdir(parents=True,exist_ok=True)
OUT=BASE_OUT
REF=ROOT/'Outputs/V3/01_reference_data_nadine34'
A=json.loads((ROOT/'Outputs/V3/00_input_audit/00_input_manifest.json').read_text())
assert A.get('schema_version')=='v3_input_audit_5_nadine_circles'
assert A['exhaustive_annotations'].get('all_annotation_polygons_musanga') is True
assert int(A['exhaustive_annotations']['verified_exhaustive_circle_count'])==34
EXCLUDED_MOSAIC='Mos_3_4_transparent_mosaic_group1_georef.tif'
mosaics={m['file']:m for m in A['mosaics'] if m['file']!=EXCLUDED_MOSAIC}
assert EXCLUDED_MOSAIC in [m['file'] for m in A['mosaics']]
splits=pd.read_csv(REF/'01_cv_window_manifest.csv')
links_all=pd.read_csv(REF/'01_circle_spatial_folds.csv')
links=links_all[links_all.mosaic!=EXCLUDED_MOSAIC].copy()
EXCLUDED_IDS=sorted(set(links_all.circle_id.astype(int))-set(links.circle_id.astype(int)))
assert len(EXCLUDED_IDS)==4,(EXCLUDED_IDS,len(links))
assert len(links)==30 and links.circle_id.nunique()==30
print('Excluding MOS 3-4 circle IDs:',EXCLUDED_IDS)
repairs=pd.read_csv(REF/'01_geometry_repairs.csv')
assert links.circle_id.nunique()==30
assert links.groupby('site').circle_id.nunique().to_dict()=={'Baego':7,'MOS':11,'Yangambi':7,'Yoko':5}
FOLD=0;SEED=20261016;CROP=256;EPOCHS=25;BATCH=8
FOLDS=[0,1,2]
TREATMENTS=['native']  # faster diagnostic, 6 fits; optionally append 'augment','v2_E1_site'
TRAIN_MODELS=['M1','M2']
MAX_PARTIAL_PER_MOSAIC=250
MAX_EXHAUSTIVE_PER_SITE=260
HOLDOUT_COUNTS={'Baego':1,'MOS':2,'Yangambi':2,'Yoko':2}
assert torch.cuda.is_available(),'Select Runtime > Change runtime type > GPU for the pilot.'
device='cuda';torch.manual_seed(SEED);np.random.seed(SEED);random.seed(SEED)
print('GPU',torch.cuda.get_device_name(0),'| fold',FOLD)
print('Train/val candidates:',splits[splits.cv_fold==FOLD].split.value_counts().to_dict())
print('Files:',ROOT,'| output:',OUT)




print('30-circle NB02D: excluding MOS 3-4 from training and evaluation; latest annotations used.')
print('Only',len(mosaics),'mosaics loaded; held-out circles:',len(links))


## 02 — Read and validate annotated geometry (same definitions as 02C)

In [ ]:
def pieces(g):
    if g.geom_type in ('Polygon','MultiPolygon'):return [g]
    if hasattr(g,'geoms'):return [p for q in g.geoms for p in pieces(q)]
    return []
def polygons(path):
    src=gpd.read_file(path);inds=[];shapes=[]
    for i,g in enumerate(src.geometry):
        if g is None or g.is_empty:continue
        if not g.is_valid:
            try:
                fixed=unary_union(pieces(make_valid(g)))
                change=abs(fixed.area-g.area)/max(fixed.area,g.area,1e-12)
                if change>(.01 if Path(path) in (Path(A['exhaustive_annotations']['complete_roi_boundaries']),Path(A['exhaustive_annotations']['positive_polygons']),Path(A['roi'])) else .05) or not fixed.is_valid:continue
                g=fixed
            except Exception:continue
        inds.append(i);shapes.append(g)
    x=src.iloc[inds].copy().reset_index(drop=True)
    x.geometry=shapes
    x['original_row']=inds
    return x
roi=polygons(A['roi'])
circles=polygons(A['exhaustive_annotations']['complete_roi_boundaries'])
positive=polygons(A['exhaustive_annotations']['positive_polygons'])
assert len(circles)==34 and len(positive)>0
print('Current positive annotation polygons:',len(positive),'(not hard-coded to historical count)')
circle_ids=set(links.circle_id.astype(int))
circles=circles[circles.original_row.isin(circle_ids)].copy()
tree={}
for site,path in A['shared_annotations'].items():
    t=polygons(path)
    cls=pd.to_numeric(t.Class_ID,errors='coerce')
    t['class01']=np.where(cls.eq(1),1,np.where(cls.notna(),0,-1))
    tree[site]=t
geo={}
for name,m in mosaics.items():
    with rasterio.open(m['path']) as src:
        c=circles.to_crs(src.crs).copy()
        c['fold']=c.original_row.map(links.set_index('circle_id').fold)
        c['owner']=c.original_row.map(links.set_index('circle_id').mosaic)
        geo[name]=dict(site=m['site'],path=m['path'],transform=src.transform,
            crs=src.crs,roi=roi.to_crs(src.crs),
            circles=c,positive=positive.to_crs(src.crs),trees=tree[m['site']].to_crs(src.crs))



In [ ]:
class TinyUNet(nn.Module):
    def __init__(self,base=16):
        super().__init__()
        def block(a,b):
            return nn.Sequential(nn.Conv2d(a,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU(inplace=True),
                nn.Conv2d(b,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU(inplace=True))
        self.e1=block(3,base);self.e2=block(base,base*2)
        self.bridge=block(base*2,base*4)
        self.d2=block(base*4+base*2,base*2)
        self.d1=block(base*2+base,base)
        self.final=nn.Conv2d(base,1,1)
    def forward(self,x):
        e1=self.e1(x);e2=self.e2(F.max_pool2d(e1,2))
        b=self.bridge(F.max_pool2d(e2,2))
        x=self.d2(torch.cat([F.interpolate(b,size=e2.shape[-2:],mode='bilinear',align_corners=False),e2],1))
        x=self.d1(torch.cat([F.interpolate(x,size=e1.shape[-2:],mode='bilinear',align_corners=False),e1],1))
        return self.final(x)[:,0]
class CropData(Dataset):
    def __init__(self,rows,augment=False):self.rows=rows;self.augment=augment
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        d=self.rows[i];image=d['rgb'];y=d['y'];w=d['w']
        if self.augment:
            rng=np.random.default_rng(SEED+i+int(torch.randint(0,1000000,(1,)).item()))
            k=int(rng.integers(4));image=np.rot90(image,k);y=np.rot90(y,k);w=np.rot90(w,k)
            if rng.random()<.5:image=image[:,::-1];y=y[:,::-1];w=w[:,::-1]
        x=torch.tensor(np.ascontiguousarray(image.transpose(2,0,1)),dtype=torch.float32)
        return x,torch.tensor(np.ascontiguousarray(y)),torch.tensor(np.ascontiguousarray(w))
def masked_loss(logits,y,w):
    loss=F.binary_cross_entropy_with_logits(logits,y,reduction='none')
    return (loss*w).sum()/w.sum().clamp_min(1.)
print('Tiny U-Net ready (native 256×256 RGB crops).')




## 03 — Fold-specific masks and candidate windows

In [ ]:
from rasterio.windows import Window
from torch.utils.data import Dataset,DataLoader
candidate=pd.read_csv(REF/'01_candidate_windows.csv')
all_circle_buffers={}
BLOCKED={}
DEV=set();HOLDOUT=set()
for name,d in geo.items():
    cs=circles.to_crs(d['crs'])
    all_circle_buffers[name]=[g.buffer(15.) for g in cs.geometry]
    d['dev_circles']=cs[cs.original_row.isin(DEV)]
    d['holdout_circles']=cs[cs.original_row.isin(HOLDOUT)]
    d['positive_local']=positive.to_crs(d['crs'])
    d['trees_local']=tree[d['site']].to_crs(d['crs')]
    d['roi_local']=roi.to_crs(d['crs'])

# Local SSD staging: copy each retained mosaic at most once per runtime.
# Source image metadata used for the cache fingerprint remains the original Drive path.
import shutil
STAGE_DIR=Path('/content/musanga_02d_30_stage');STAGE_DIR.mkdir(parents=True,exist_ok=True)
USE_LOCAL_RGB=True
for name,d in geo.items():
    source=Path(d['path'])
    free=shutil.disk_usage('/content').free
    dest=STAGE_DIR/name
    if not USE_LOCAL_RGB or free < source.stat().st_size + 3*1024**3:
        print('SSD stage skipped (insufficient space or disabled):',name,flush=True)
        continue
    if not dest.exists() or dest.stat().st_size!=source.stat().st_size:
        temp=dest.with_name(dest.name+'.part')
        print('STAGING',name,round(source.stat().st_size/1e9,2),'GB',flush=True)
        started=time.perf_counter()
        try:
            with source.open('rb') as a,temp.open('wb') as b:
                shutil.copyfileobj(a,b,length=16*1024*1024)
            assert temp.stat().st_size==source.stat().st_size
            temp.replace(dest)
            print('STAGED in',round(time.perf_counter()-started,1),'seconds',flush=True)
        except Exception as exc:
            print('STAGING FAILED, using Drive:',repr(exc),flush=True)
            temp.unlink(missing_ok=True)
            continue
    d['path']=str(dest)
    with rasterio.open(dest) as staged:
        assert staged.crs==d['crs'] and staged.transform==d['transform']
print('Local SSD staging done; retained mosaics:',list(geo))

def burn_window(geometries,name,win):
    """GeoSeries or GeoDataFrame -> Shapely geometries only."""
    d=geo[name]
    series=geometries.geometry if isinstance(geometries,gpd.GeoDataFrame) else geometries
    assert isinstance(series,gpd.GeoSeries),type(series)
    bb=box(*rasterio.windows.bounds(win,d['transform']))
    ids=series.sindex.query(bb,predicate='intersects')
    if len(ids)==0:return np.zeros((int(win.height),int(win.width)),dtype=bool)
    gs=series.iloc[ids]
    return rasterize(((g,1) for g in gs.values if g is not None and not g.is_empty),
       out_shape=(int(win.height),int(win.width)),
       transform=rasterio.windows.transform(win,d['transform']),
       fill=0,dtype='uint8').astype(bool)

def rgb_for_window(name,win):
    with rasterio.open(geo[name]['path']) as ds:
        raw=ds.read([1,2,3],window=win,masked=True)
    valid=~np.any(np.ma.getmaskarray(raw),axis=0)
    a=raw.filled(0).astype('float32')
    valid &= np.any(a!=0,axis=0)&np.any(a!=255,axis=0)
    valid &= ~((a[0]==0)&(a[1]==0)&(a[2]==255)) # Baego blue nodata
    valid &= burn_window(geo[name]['roi_local'].geometry,name,win)
    return np.clip(a/255.,0,1).transpose(1,2,0).astype('float32'),valid

def training_crop(rec,model):
    name=rec.mosaic;d=geo[name]
    if rec.height<CROP or rec.width<CROP:return None
    rr=int(rec.row)+(int(rec.height)-CROP)//2
    cc=int(rec.col)+(int(rec.width)-CROP)//2
    win=Window(cc,rr,CROP,CROP)
    bb=box(*rasterio.windows.bounds(win,d['transform']))
    if any(bb.intersects(g) for g in BLOCKED[name]):return None
    if model=='M1' and any(bb.intersects(g) for g in all_circle_buffers[name]):return None
    if model=='M2' and rec.kind=='exhaustive' and int(rec.circle_id) not in DEV:return None
    inside=burn_window(d['roi_local'].geometry,name,win)
    pos_tree=burn_window(d['tree_pos'],name,win)
    neg_tree=burn_window(d['tree_neg'],name,win)
    if model=='M1':
        full=np.zeros_like(inside);pos_ex=np.zeros_like(inside)
    else:
        full=burn_window(d['dev_circles'].geometry,name,win)&inside
        pos_ex=burn_window(d['positive_local'].geometry,name,win)&full
    conflict=(pos_tree&neg_tree)| (pos_ex&neg_tree) | (full&~pos_ex&pos_tree)
    target=(pos_ex|(~full&pos_tree)).astype('float32')
    weight=(full|((pos_tree|neg_tree)&~full))&inside&~conflict
    img,valid=rgb_for_window(name,win)
    weight &= valid
    if int(weight.sum())<250:return None
    return dict(image=img,target=target,weight=weight.astype('float32'),
                site=d['site'],kind=rec.kind,mosaic=name)

def make_training_cache(model):
    selected=[]
    for (site,kind),grp in candidate.groupby(['site','kind']):
        if model=='M1' and kind=='exhaustive':continue
        if model=='M2' and kind=='exhaustive':
            grp=grp[grp.circle_id.isin(DEV)]
        limit=MAX_EXHAUSTIVE_PER_SITE if kind=='exhaustive' else MAX_PARTIAL_PER_MOSAIC*(3 if site=='MOS' else 1)
        subset=grp.sample(n=min(limit,len(grp)),random_state=SEED)
        selected.extend(subset.itertuples(index=False))
    result=[]
    prep_started=time.perf_counter()
    for i,rec in enumerate(selected):
        chip=training_crop(rec,model)
        if chip is not None:result.append(chip)
        if (i+1)%100==0:
            print(f"{model}: prepared {i+1}/{len(selected)} candidates, usable {len(result)}, elapsed {time.perf_counter()-prep_started:.1f}s",flush=True)
    site_counts = pd.Series([item["site"] for item in result]).value_counts().to_dict()
    print(f"{model}: usable training chips = {len(result)}", flush=True)
    print("Training chips by site:", site_counts, flush=True)
    print(f"{model}: cache preparation took {time.perf_counter()-prep_started:.1f}s",flush=True)
    assert len(result)>15,f'{model}: too few chips'
    return result

class CropDataset(Dataset):
    def __init__(self,rows):self.rows=rows
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        q=self.rows[i];x=q['image'].copy();y=q['target'].copy();w=q['weight'].copy()
        k=int(torch.randint(0,4,(1,)).item())
        x=np.rot90(x,k);y=np.rot90(y,k);w=np.rot90(w,k)
        if torch.rand(1).item()<.5:x=x[:,::-1];y=y[:,::-1];w=w[:,::-1]
        return (torch.from_numpy(np.ascontiguousarray(x.transpose(2,0,1))),
                torch.from_numpy(np.ascontiguousarray(y)),torch.from_numpy(np.ascontiguousarray(w)))


def configure_fold(fold):
    global FOLD,DEV,HOLDOUT,BLOCKED,split_df
    FOLD=int(fold)
    HOLDOUT=set(links.loc[links.fold==fold,'circle_id'].astype(int))
    DEV=set(links.circle_id.astype(int))-HOLDOUT
    assert len(HOLDOUT)>0 and len(DEV)>0
    split_df=links[['circle_id','site','mosaic','cover_pct']].copy()
    split_df['split']=np.where(split_df.circle_id.isin(HOLDOUT),'holdout','development')
    BLOCKED={}
    for name,d in geo.items():
        cs=circles.to_crs(d['crs'])
        d['dev_circles']=cs[cs.original_row.isin(DEV)]
        d['holdout_circles']=cs[cs.original_row.isin(HOLDOUT)]
        group_ids=set(links[(links.site==d['site'])&links.circle_id.isin(HOLDOUT)].circle_id.astype(int))
        BLOCKED[name]=[g.buffer(15.) for g in cs[cs.original_row.isin(group_ids)].geometry]
    assert split_df.query("split=='holdout'").site.nunique()==4
    print('Fold',fold,'held-out circles',sorted(HOLDOUT),flush=True)


## 04 — V2 E1 normalization: unsupervised fixed site luminance gains

Recovered from `V2/03_UAV_Illumination_Harmonization.ipynb`: site median luminance, reference median of the four training forest groups, global gain bounds **(0.85, 1.15)**, valid-pixel mask and unchanged original TIFFs. Samples are dispersed across each native mosaic; MOS is grouped as one forest. Site statistics use imagery only, not reference labels. The V2 local E2/E3 methods are deliberately excluded from this first controlled comparison.
**Reconnects:** Section 04 loads the saved E1 site statistics from Drive if available; new caches are input-fingerprint checked. Existing older corrected 02D caches can also be reused (with a warning because they lack fingerprints). Set `FORCE_RECOMPUTE_ILLUM=True` inside the code cell if you changed the ROI or mosaics after generating a legacy cache.


In [ ]:
from rasterio.windows import Window
from scipy.stats import linregress
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error
RGB_TREATMENT='native'
_ORIG_RGB=rgb_for_window
def luminance(x):
    return .2126*x[:,:,0]+.7152*x[:,:,1]+.0722*x[:,:,2]
def valid_roi_rgb_sample(name,src,win):
    """Area-limited sampling: raster-mask + RGB nodata + authoritative ROI."""
    raw=src.read([1,2,3],window=win,masked=True)
    rgb=raw.filled(0)
    valid=~np.any(np.ma.getmaskarray(raw),axis=0)
    valid &= np.any(rgb!=0,axis=0)&np.any(rgb!=255,axis=0)
    valid &= ~((rgb[0]==0)&(rgb[1]==0)&(rgb[2]==255))
    # Respect alpha band only if declared as such; do not guess band 4 semantics.
    if src.count>=4 and any(ci.name=='alpha' for ci in src.colorinterp):
        ai=next(i+1 for i,ci in enumerate(src.colorinterp) if ci.name=='alpha')
        valid &= src.read(ai,window=win)>0
    valid &= burn_window(geo[name]['roi_local'].geometry,name,win)
    a=np.moveaxis(rgb,0,-1).astype('float32')/255.
    return a,valid
def sample_mosaic_luminance(name,n=80,side=256):
    rng=np.random.default_rng(SEED+sum(ord(q) for q in name))
    values=[]
    d=geo[name]
    with rasterio.open(d['path']) as src:
        for i in range(n):
            # Stratified, seeded, spread over image extent; exclude nodata and nearly empty patches
            fraction=(i+.5)/n
            row=int(np.clip(((i%9)+rng.random())/9*max(0,src.height-side),0,max(0,src.height-side)))
            col=int(np.clip(((i//9)+rng.random())/max(1,int(np.ceil(n/9)))*max(0,src.width-side),
                            0,max(0,src.width-side)))
            a,valid=valid_roi_rgb_sample(name,src,Window(col,row,side,side))
            if valid.mean()<.8:continue
            values.append(float(np.median(luminance(a)[valid])))
    if len(values)<3:
        # Sparse valid footprints: use already-audited candidate-window locations
        # as an image-only sampling fallback. No reference labels are read.
        fallback=candidate[candidate.mosaic==name]
        with rasterio.open(d['path']) as src:
            for row in fallback.sample(n=min(60,len(fallback)),random_state=SEED).itertuples(index=False):
                rr=int(row.row);cc=int(row.col)
                h=min(side,src.height-rr);w=min(side,src.width-cc)
                if h<=0 or w<=0:continue
                a,valid=valid_roi_rgb_sample(name,src,Window(cc,rr,w,h))
                if valid.mean()>=.8:values.append(float(np.median(luminance(a)[valid])))
                if len(values)>=10:break
    if len(values)<3:
        raise RuntimeError(f'{name}: insufficient valid RGB even after annotated-window fallback ({len(values)})')
    return float(np.median(values)),len(values)
# Reuse expensive ROI-masked illumination audit when inputs and method match.
# Backwards-compatible with earlier 02D_E1_site_gains.json (no fingerprint).
import hashlib
ILLUM_FILE=BASE_OUT/'02D_E1_site_gains.json'
ILLUM_METHOD='v2_E1_site_roi_rgbmask_v2'
def illumination_inputs_signature():
    def stat(p):
        p=Path(p)
        return dict(path=str(p.resolve()),bytes=p.stat().st_size,
                    mtime_ns=p.stat().st_mtime_ns)
    mosaics_signature={name:stat(geo[name]['path']) for name in sorted(geo)}
    # Shapefile sidecars affect polygon geometry and projection.
    roi_path=Path(A['roi'])
    roi_signature={ext:stat(roi_path.with_suffix(ext))
                   for ext in ('.shp','.shx','.dbf','.prj')
                   if roi_path.with_suffix(ext).exists()}
    return dict(mosaics=mosaics_signature,roi=roi_signature,seed=int(SEED),
                sample_n=80,side_px=256,valid_fraction_min=.8,
                method=ILLUM_METHOD,gain_bounds=[.85,1.15])
signature=illumination_inputs_signature()
def valid_saved_illumination(data):
    if not isinstance(data,dict):return False
    if data.get('v2_method')!='E1_site':return False
    sites={'Baego','MOS','Yangambi','Yoko'}
    if set(data.get('site_luminance',{}))!=sites or set(data.get('site_gain',{}))!=sites:
        return False
    if set(data.get('mosaic_candidates',{}))!=set(geo):return False
    if list(data.get('gain_bounds',[]))!=[.85,1.15]:return False
    try:
        lum=[float(v) for v in data['site_luminance'].values()]
        gains=[float(v) for v in data['site_gain'].values()]
        if not all(np.isfinite(v) and .15<=v<=.85 for v in lum):return False
        if not all(np.isfinite(v) and .85<=v<=1.15 for v in gains):return False
        if not all(int(v['n'])>=3 for v in data['mosaic_candidates'].values()):return False
    except (ValueError,TypeError,KeyError,OverflowError):return False
    return True

illum_saved=None
if ILLUM_FILE.exists():
    try:
        illum_candidate=json.loads(ILLUM_FILE.read_text())
        if valid_saved_illumination(illum_candidate):
            if illum_candidate.get('input_signature')==signature:
                illum_saved=illum_candidate
                print('LOADED saved E1 luminance statistics (current inputs verified):',ILLUM_FILE)
            elif 'input_signature' not in illum_candidate:
                # Older corrected 02D run: usable as a one-time migration.
                # Old cache cannot prove ROI/mosaic files are unchanged; show warning.
                illum_saved=illum_candidate
                print('LOADED legacy E1 luminance cache; input fingerprints were not recorded.')
                print('If ROI/mosaics have changed since this cache was created, set FORCE_RECOMPUTE_ILLUM=True.')
            else:
                print('Illumination cache inputs changed; recomputing.')
        else:
            print('Saved illumination statistics invalid or incomplete; recomputing.')
    except (ValueError,OSError,TypeError) as e:
        print('Unreadable illumination cache; recomputing:',str(e))
FORCE_RECOMPUTE_ILLUM=False  # Set True only if you need fresh illumination sampling.
if FORCE_RECOMPUTE_ILLUM:
    illum_saved=None
if illum_saved is None:
    print('Sampling native RGB within ROI for E1 illumination statistics...',flush=True)
    MOSAIC_LUM={name:sample_mosaic_luminance(name) for name in geo}
    SITE_LUM={}
    for site in ['Baego','MOS','Yangambi','Yoko']:
        values=[value for name,(value,n) in MOSAIC_LUM.items()
                if geo[name]['site']==site]
        SITE_LUM[site]=float(np.median(values))
    REF_LUM=float(np.median(list(SITE_LUM.values())))
    SITE_GAIN={site:float(np.clip(REF_LUM/v,.85,1.15)) for site,v in SITE_LUM.items()}
    payload=dict(v2_method='E1_site',gain_bounds=[.85,1.15],
                 reference_luminance=REF_LUM,site_luminance=SITE_LUM,
                 site_gain=SITE_GAIN,
                 mosaic_candidates={k:dict(median=v[0],n=v[1])
                                    for k,v in MOSAIC_LUM.items()},
                 input_signature=signature)
    # Only publish successfully completed results, never partial cache content.
    tmp=ILLUM_FILE.with_suffix('.json.tmp')
    tmp.write_text(json.dumps(payload,indent=2))
    tmp.replace(ILLUM_FILE)
    print('SAVED E1 luminance statistics:',ILLUM_FILE,flush=True)
else:
    SITE_LUM={k:float(v) for k,v in illum_saved['site_luminance'].items()}
    SITE_GAIN={k:float(v) for k,v in illum_saved['site_gain'].items()}
    REF_LUM=float(illum_saved['reference_luminance'])
    MOSAIC_LUM={k:(float(v['median']),int(v['n']))
                for k,v in illum_saved['mosaic_candidates'].items()}
display(pd.DataFrame([dict(site=site,luminance=SITE_LUM[site],gain=SITE_GAIN[site])
                      for site in SITE_LUM]))
def rgb_for_window(name,win):
    rgb,valid=_ORIG_RGB(name,win)
    if RGB_TREATMENT=='v2_E1_site':
        rgb=np.clip(rgb*SITE_GAIN[geo[name]['site']],0,1).astype('float32')
    return rgb,valid


## 05 — Cached fold-specific M1/M2 training: native, augmented or V2 E1

Identical label policy and training epochs. Augmented treatment uses stochastic brightness, contrast and gamma on the RGB input only; target and supervision masks are unchanged. Model checkpoints written per fold/treatment to Drive.

In [ ]:
class IlluminationCropDataset(CropDataset):
,    def __init__(self,rows,mode):
,        super().__init__(rows);self.mode=mode
,    def __getitem__(self,i):
,        image,y,w=super().__getitem__(i)
,        if self.mode=='v2_E1_site':
,            image=torch.clamp(image*SITE_GAIN[self.rows[i]['site']],0,1)
,        if self.mode=='augment':
,            b=float(torch.empty(1).uniform_(.80,1.20))
,            contrast=float(torch.empty(1).uniform_(.85,1.15))
,            gamma=float(torch.empty(1).uniform_(.8,1.2))
,            image=torch.clamp((image-.5)*contrast+.5,0,1)
,            image=torch.clamp(image*b,0,1)**gamma
,        return image,y,w
,import hashlib
,CACHE_VERSION='02D_30noMOS34_reannotated_v2'
,CACHE_DIR=BASE_OUT/'training_chip_cache'
,CACHE_DIR.mkdir(parents=True,exist_ok=True)
,
,def _file_meta(path):
,    p=Path(path)
,    assert p.exists(),f'Missing input: {p}'
,    return [str(p.resolve()),int(p.stat().st_size),int(p.stat().st_mtime_ns)]
,
,def _vector_meta(path):
,    p=Path(path)
,    if p.suffix.lower()!='.shp':return [_file_meta(p)]
,    return [_file_meta(p.with_suffix(ext)) for ext in
,            ('.shp','.shx','.dbf','.prj','.cpg')
,            if p.with_suffix(ext).exists()]
,
,def chip_cache_key(label):
,    deps=[_file_meta(REF/'01_candidate_windows.csv'),
,          _file_meta(REF/'01_circle_spatial_folds.csv'),
,          _file_meta(ROOT/'Outputs/V3/00_input_audit/00_input_manifest.json')]
,    for p in [A['roi'],A['exhaustive_annotations']['complete_roi_boundaries'],
,              A['exhaustive_annotations']['positive_polygons'],
,              *[A['shared_annotations'][site] for site in sorted(A['shared_annotations'])]]:
,        deps.extend(_vector_meta(p))
,    deps.extend(_file_meta(mosaics[name]['path']) for name in sorted(mosaics))
,    info=dict(version=CACHE_VERSION,dependencies=deps,fold=int(FOLD),
,        model=label,train_circle_ids=sorted(int(i) for i in DEV),
,        holdout_circle_ids=sorted(int(i) for i in HOLDOUT),
,        seed=int(SEED),crop=CROP,max_partial=MAX_PARTIAL_PER_MOSAIC,
,        max_exhaustive=MAX_EXHAUSTIVE_PER_SITE,
,        full_conflicts_zero_weight=True,minimum_weight_pixels=250)
,    digest=hashlib.sha256(json.dumps(info,sort_keys=True).encode()).hexdigest()
,    return digest
,
,def cached_training_chips(label):
,    key=chip_cache_key(label)
,    path=CACHE_DIR/f'{label}_fold{FOLD}_{key[:20]}.npz'
,    if path.exists():
,        started=time.perf_counter()
,        print('CACHE HIT',label,'fold',FOLD,path.name,flush=True)
,        try:
,            with np.load(path,allow_pickle=False) as z:
,                assert str(z['fingerprint'].item())==key
,                rgb=z['rgb'];y=z['y'];w=z['w']
,                sites=z['site'].astype(str).tolist()
,                kinds=z['kind'].astype(str).tolist()
,                assert rgb.shape[1:]==(CROP,CROP,3)
,                assert rgb.shape[:3]==y.shape==w.shape
,                assert len(rgb)==len(sites)==len(kinds)>15
,                rows=[dict(image=rgb[i].astype('float32')/255.,
,                           target=y[i].astype('float32'),
,                           weight=w[i].astype('float32'),
,                           site=sites[i],kind=kinds[i])
,                      for i in range(len(rgb))]
,            print(f'Loaded {len(rows)} chips in {time.perf_counter()-started:.1f}s',flush=True)
,            return rows
,        except Exception as exc:
,            print('Invalid cache; rebuilding:',repr(exc),flush=True)
,            path.unlink(missing_ok=True)
,    # Always produce native RGB chips. Site E1 gain is applied IN THE TRAINING
,    # DATASET after loading, so the identical native chips are reused across
,    # all three RGB treatments without contaminating the cache.
,    global RGB_TREATMENT
,    original_treatment=RGB_TREATMENT
,    RGB_TREATMENT='native'
,    try:
,        started=time.perf_counter()
,        print('CACHE MISS:',label,'fold',FOLD,'building native source chips...',flush=True)
,        rows=make_training_cache(label)
,    finally:
,        RGB_TREATMENT=original_treatment
,    print(f'Prepared {len(rows)} chips in {time.perf_counter()-started:.1f}s',flush=True)
,    temp=path.with_name(path.name+'.tmp.npz')
,    try:
,        np.savez_compressed(temp,fingerprint=np.array(key),
,            rgb=np.stack([np.rint(np.clip(x['image'],0,1)*255).astype('uint8') for x in rows]),
,            y=np.stack([x['target'].astype('uint8') for x in rows]),
,            w=np.stack([x['weight'].astype('uint8') for x in rows]),
,            site=np.array([x['site'] for x in rows],dtype='U12'),
,            kind=np.array([x['kind'] for x in rows],dtype='U16'))
,        temp.replace(path)
,    finally:
,        temp.unlink(missing_ok=True)
,    print('SAVED CACHE',path.name,round(path.stat().st_size/1e6,1),'MB',flush=True)
,    return rows
,
,import time
,def fit_network(label):
,    ckpt=OUT/f'{label}_model.pt'
,    torch.manual_seed(SEED)
,    model=TinyUNet().to(device)
,    if ckpt.exists():
,        model.load_state_dict(torch.load(ckpt,map_location=device,weights_only=True))
,        print('Loaded',ckpt,flush=True)
,        return model
,    cache=cached_training_chips(label)
,    print(f"{label} fold {FOLD} treatment {RGB_TREATMENT}: "
,          f"{len(cache)} chips, {EPOCHS} epochs",flush=True)
,    loader=DataLoader(IlluminationCropDataset(cache,RGB_TREATMENT),
,        batch_size=BATCH,shuffle=True,num_workers=0,pin_memory=True)
,    optim=torch.optim.AdamW(model.parameters(),lr=7e-4,weight_decay=1e-4)
,    scaler=torch.amp.GradScaler('cuda',enabled=True)
,    history=[]
,    for epoch in range(EPOCHS):
,        model.train();losses=[];start=time.perf_counter()
,        for x,y,w in loader:
,            x=x.to(device,non_blocking=True)
,            y=y.to(device,non_blocking=True)
,            w=w.to(device,non_blocking=True)
,            optim.zero_grad(set_to_none=True)
,            with torch.autocast(device_type='cuda',dtype=torch.float16):
,                logits=model(x)
,            loss=masked_loss(logits.float(),y,w)
,            scaler.scale(loss).backward()
,            scaler.step(optim)
,            scaler.update()
,            losses.append(float(loss.detach()))
,        history.append(dict(epoch=epoch+1,masked_BCE=float(np.mean(losses)),
,                            epoch_seconds=time.perf_counter()-start))
,        pd.DataFrame(history).to_csv(OUT/f'{label}_loss.csv',index=False)
,        print(f'{RGB_TREATMENT} fold {FOLD} {label} '
,              f'epoch {epoch+1:02d}/{EPOCHS}: BCE={np.mean(losses):.4f} '
,              f'{time.perf_counter()-start:.1f}s',flush=True)
,    torch.save(model.state_dict(),ckpt)
,    del cache
,    return model


## 06 — Full-circle predicted fractions; all pixels inside held-out circles

Avoid the previous fixed 7-circle split. Validate every exhaustive circle exactly once per variant; RGB remains at native resolution and no background outside the circle is assumed labelled.

In [ ]:
from scipy.special import expit,logit
,from scipy.optimize import minimize
,from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error
,WINDOWS=pd.read_csv(REF/'01_candidate_windows.csv')
,VAL_WINDOWS=WINDOWS[WINDOWS.kind=='exhaustive'].copy()
,VAL_WINDOWS=VAL_WINDOWS[VAL_WINDOWS.circle_id.isin(links.circle_id)].copy()
assert set(VAL_WINDOWS.circle_id.unique().astype(int))==set(links.circle_id.astype(int))
,N_BINS=256
,BINS=np.linspace(0,1,N_BINS+1)
,BIN_CENTER=(BINS[:-1]+BINS[1:])/2
,
,def run_all_circles(model,model_name):
,    statfile=OUT/f'{model_name}_circle_raw.csv'
,    histfile=OUT/f'{model_name}_histograms.npz'
,    if statfile.exists() and histfile.exists():
,        return pd.read_csv(statfile),dict(np.load(histfile))
,    model.eval();rows=[];histograms={}
,    for _,circle in split_df[split_df.split=='holdout'].iterrows():
,        cid=int(circle.circle_id)
,        name=circle.mosaic
,        d=geo[name]
,        windows=VAL_WINDOWS[(VAL_WINDOWS.circle_id==cid)&(VAL_WINDOWS.mosaic==name)]
,        assert len(windows),f'No evaluation windows for circle {cid}'
,        cg=circles[circles.original_row==cid].to_crs(d['crs'])
,        counts=np.zeros(N_BINS,np.int64);true_counts=np.zeros(N_BINS,np.int64)
,        n=0;sumy=0.;sump=0.;sum_sq=0.;tp=fp=fn=0
,        rng=np.random.default_rng(SEED+cid)
,        sample_truth=[];sample_probs=[]
,        for rec in windows.itertuples(index=False):
,            win=Window(int(rec.col),int(rec.row),int(rec.width),int(rec.height))
,            mask=burn_window(cg.geometry,name,win)
,            mask &= burn_window(d['roi_local'].geometry,name,win)
,            rgb,valid=rgb_for_window(name,win)
,            mask &= valid
,            if not mask.any():continue
,            reference=burn_window(d['positive_local'].geometry,name,win)
,            x=torch.from_numpy(rgb.transpose(2,0,1).copy())[None].to(device)
,            with torch.inference_mode():
,                pred=torch.sigmoid(model(x))[0].cpu().numpy()
,            y=reference[mask].astype('uint8');p=pred[mask].astype('float32')
,            n+=len(y);sumy+=float(y.sum());sump+=float(p.sum())
,            sum_sq+=float(np.square(p-y).sum())
,            binary=p>=.5
,            tp+=int(np.sum(binary&(y==1)));fp+=int(np.sum(binary&(y==0)))
,            fn+=int(np.sum((~binary)&(y==1)))
,            ix=np.minimum((p*N_BINS).astype(np.int32),N_BINS-1)
,            counts+=np.bincount(ix,minlength=N_BINS).astype('int64')
,            true_counts+=np.bincount(ix,weights=y,minlength=N_BINS).astype('int64')
,            # only diagnostic AUC/AP uses pixel subsampling, with fixed per-circle cap
,            take=rng.choice(len(y),min(100,len(y)),replace=False)
,            sample_truth.extend(y[take].tolist());sample_probs.extend(p[take].tolist())
,        assert n>0,f'No valid ROI/RGB reference pixels for circle {cid}'
,        assert int(counts.sum())==n
,        yt=np.asarray(sample_truth);pr=np.asarray(sample_probs)
,        rows.append(dict(model=model_name,circle_id=cid,site=circle.site,split=circle.split,
,                         valid_pixels=n,observed_fraction=100*sumy/n,
,                         predicted_fraction=100*sump/n,
,                         bias_pp=100*(sump-sumy)/n,brier=sum_sq/n,
,                         f1_threshold_05=2*tp/max(1,2*tp+fp+fn),
,                         auc_sample=roc_auc_score(yt,pr) if len(np.unique(yt))>1 else np.nan,
,                         ap_sample=average_precision_score(yt,pr) if len(np.unique(yt))>1 else np.nan))
,        histograms[str(cid)+'_count']=counts
,        histograms[str(cid)+'_positive']=true_counts
,        print(model_name,'circle',cid,'obs',round(100*sumy/n,1),
,              'pred',round(100*sump/n,1),'valid pixels',n,flush=True)
,    frame=pd.DataFrame(rows)
,    assert len(frame)==len(HOLDOUT)
,    frame.to_csv(statfile,index=False)
,    np.savez_compressed(histfile,**histograms)
,    return frame,histograms
,


## 07 — Run/resume 18 fits and held-out evaluation

A complete (fold, treatment, model) pair has both a model checkpoint and a circle-score CSV. Completed runs are not retrained. We run M1 and M2, but M3 calibration is deliberately not part of the main benchmark after its negative result in 02C.

In [ ]:
import gc
for treatment in TREATMENTS:
    RGB_TREATMENT=treatment
    for fold in FOLDS:
        configure_fold(fold)
        OUT=BASE_OUT/treatment/f'fold_{fold}'
        OUT.mkdir(parents=True,exist_ok=True)
        for label in TRAIN_MODELS:
            scorefile=OUT/f'{label}_circle_raw.csv'
            checkpoint=OUT/f'{label}_model.pt'
            if scorefile.exists() and checkpoint.exists():
                print('SKIP completed',treatment,fold,label,flush=True)
                continue
            print('\nSTART',treatment,'fold',fold,label,flush=True)
            model=fit_network(label)
            score,_=run_all_circles(model,label)
            assert set(score.circle_id)==HOLDOUT,('Missing holdouts',treatment,fold,label)
            print('DONE',treatment,fold,label,'circles',len(score),flush=True)
            del model
            torch.cuda.empty_cache()
            gc.collect()
print('All scheduled CV experiments completed or resumed.')


## 08 — Observed-versus-predicted across all 34 independently held-out circles

Model selection criterion is **site-macro circle MAE**, supported by 22-circle R², RMSE, bias, calibration slope/intercept and Brier. A site-macro criterion prevents the site with seven circles from automatically outweighing Baego with only three.

In [ ]:
results=[]
,for treatment in TREATMENTS:
,    for fold in FOLDS:
,        for label in TRAIN_MODELS:
,            file=BASE_OUT/treatment/f'fold_{fold}'/f'{label}_circle_raw.csv'
,            assert file.exists(),f'Missing {file}: finish Section 07 first'
,            q=pd.read_csv(file)
,            q['treatment']=treatment;q['fold']=fold
,            results.append(q)
,CV=pd.concat(results,ignore_index=True)
,assert CV.groupby(['model','treatment']).circle_id.nunique().eq(30).all()
,def metrics(q):
,    y=q.observed_fraction.to_numpy(float);pred=q.predicted_fraction.to_numpy(float)
,    lr=linregress(y,pred)
,    return pd.Series(dict(n_circles=len(q),r2=r2_score(y,pred),
,        pearson_r=lr.rvalue,slope=lr.slope,intercept_pp=lr.intercept,
,        MAE_pp=mean_absolute_error(y,pred),
,        RMSE_pp=np.sqrt(mean_squared_error(y,pred)),
,        bias_pp=float(np.mean(pred-y)),brier=float(q.brier.mean())))
,SUMMARY=CV.groupby(['model','treatment']).apply(metrics,include_groups=False).reset_index()
,by_site=CV.groupby(['model','treatment','site']).agg(
,    n=('circle_id','nunique'),MAE_pp=('bias_pp',lambda x:float(x.abs().mean())),
,    bias_pp=('bias_pp','mean')).reset_index()
,sm=by_site.groupby(['model','treatment']).MAE_pp.mean().rename('site_macro_MAE_pp').reset_index()
,SUMMARY=SUMMARY.merge(sm,on=['model','treatment']).sort_values(['site_macro_MAE_pp','MAE_pp'])
,display(SUMMARY.round(3))
,display(by_site.round(3))
,CV.to_csv(BASE_OUT/'02D_all_heldout_circle_predictions.csv',index=False)
,SUMMARY.to_csv(BASE_OUT/'02D_model_selection_summary.csv',index=False)
,by_site.to_csv(BASE_OUT/'02D_by_site.csv',index=False)
,fig,axs=plt.subplots(2,3,figsize=(15,10),sharex=True,sharey=True,constrained_layout=True)
,sites=['Baego','MOS','Yangambi','Yoko'];colors=dict(zip(sites,['#c27e24','#357ebd','#468c50','#944b99']))
,for i,label in enumerate(TRAIN_MODELS):
,    for j,treatment in enumerate(TREATMENTS):
,        ax=axs[i,j];q=CV[(CV.model==label)&(CV.treatment==treatment)]
,        ax.plot([0,100],[0,100],linestyle='--',color='gray',lw=1)
,        for row in q.itertuples(index=False):
,            ax.scatter(row.observed_fraction,row.predicted_fraction,
,                       color=colors[row.site],s=34)
,        mm=SUMMARY[(SUMMARY.model==label)&(SUMMARY.treatment==treatment)].iloc[0]
,        ax.set(title=f'{label}, {treatment}\nR²={mm.r2:.2f}, MAE={mm.MAE_pp:.1f} pp',xlim=(0,100),ylim=(0,100))
,        ax.grid(alpha=.15)
,for ax in axs[-1,:]:ax.set_xlabel('Observed fraction (%)')
,for ax in axs[:,0]:ax.set_ylabel('Mean predicted probability (%)')
,fig.savefig(BASE_OUT/'02D_30circle_observed_vs_predicted.png',dpi=180)
,display(fig);plt.close(fig)
,print('The best strategy is reported for 02E; holdout-site LOSO results remain separate.')


## 09 — Export selected-strategy contract for LOSO Notebook 02E

Choosing a method using all four forests' within-site CV may introduce mild selection optimism for subsequently reporting LOSO. Treat 02E as a transferability diagnostic unless selection is nested independently within its three training sites. This contract freezes the choice before running the four LOSO models.

In [ ]:
best=SUMMARY.iloc[0]
,selection=dict(source_notebook='V3/02D_30Circle_Exclude_MOS34_Revised_Annotations',selection_metric='site_macro_MAE_pp',
,  model=str(best.model),illumination=str(best.treatment),
,  site_macro_MAE_pp=float(best.site_macro_MAE_pp),all_circle_R2=float(best.r2),
,  epochs=EPOCHS,folds=FOLDS,sites=['Baego','MOS','Yangambi','Yoko'],excluded_mosaic=EXCLUDED_MOSAIC,excluded_circle_ids=EXCLUDED_IDS,
,  caution='Selected with within-site data across all 4 forests; 02E LOSO transfer metrics are descriptive')
,(BASE_OUT/'02D_selected_strategy.json').write_text(json.dumps(selection,indent=2))
,print('Selected production candidate:',selection)
